# Notebook 01: Keyword Expansion

This notebook takes initial keywords from the config and uses LLM to expand them into comprehensive search terms for ArXiv.

**Input:** `config/default.yaml` (keywords section)
**Output:** `data/keywords_expanded.csv`

**Process:**
1. Load configuration and initial keywords
2. Call LLM with expansion prompt
3. Parse JSON response into expanded keywords
4. Save to CSV with columns: `original`, `expanded`, `category`

In [ ]:
# Test package availability
import sys
import subprocess

def test_package(package_name, import_name=None):
    if import_name is None:
        import_name = package_name.replace('-', '_')
    try:
        __import__(import_name)
        print(f"✅ {package_name} available")
        return True
    except ImportError:
        print(f"❌ {package_name} missing - please run: uv sync")
        return False

# Test required packages
packages_ok = all([
    test_package('PyYAML', 'yaml'),
    test_package('openai'),
    test_package('pandas'),
    test_package('python-dotenv', 'dotenv'),
    test_package('tqdm')
])

if not packages_ok:
    print("\nPlease install missing packages and restart the notebook.")
    sys.exit(1)

In [ ]:
# Import required libraries
import os
import sys
import pandas as pd
from tqdm import tqdm

# Set project root directory (works from notebooks directory)
PROJECT_ROOT = os.path.dirname(os.getcwd())
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

# Add virtual environment packages to path
venv_path = os.path.join(PROJECT_ROOT, '.venv', 'lib', 'python3.12', 'site-packages')
sys.path.insert(0, venv_path)

# Add scripts directory to path
scripts_path = os.path.join(PROJECT_ROOT, 'scripts')
sys.path.append(scripts_path)

# Import our utilities
from utils import (
    load_config, 
    get_openai_client, 
    call_llm, 
    load_prompt_template, 
    parse_json_response,
    save_csv_checkpoint
)

print("✅ Imports successful")

In [ ]:
# Load configuration
config = load_config()
print(f"Domain: {config['domain']['name']}")
print(f"Initial keywords: {config['keywords']}")
print(f"LLM Model: {config['llm']['model']}")
print(f"Max researchers: {config['processing']['max_researchers']}")

In [ ]:
# Check for API key and initialize LLM client
try:
    client = get_openai_client(config)
    print("✅ OpenRouter client initialized")
except ValueError as e:
    print(f"❌ {e}")
    print("\nPlease:")
    print("1. Get your OpenRouter API key from https://openrouter.ai/")
    print("2. Add it to your .env file: OPENROUTER_API_KEY=your-key-here")
    print("3. Restart the notebook")
    sys.exit(1)

In [ ]:
# Load the keyword expansion prompt
prompt_template = load_prompt_template('expand_keywords')
prompt = prompt_template.format(
    domain_name=config['domain']['name'],
    domain_description=config['domain']['description'],
    keywords=config['keywords']
)

print("Prompt loaded successfully")
print(f"Prompt length: {len(prompt)} characters")

In [ ]:
# Call LLM to expand keywords
print("Calling LLM for keyword expansion...")

response = call_llm(
    client=client,
    prompt=prompt,
    model=config['llm']['model'],
    temperature=config['llm']['temperature'],
    max_tokens=config['llm']['max_tokens']
)

print("LLM response received")
print(f"Response length: {len(response)} characters")
print("\n--- Response Preview ---")
print(response[:500] + "..." if len(response) > 500 else response)

In [ ]:
# Parse the JSON response
print("Parsing LLM response...")

try:
    parsed_response = parse_json_response(response)
    print("✅ JSON parsing successful")
    
    # Check structure
    if 'expanded_keywords' not in parsed_response:
        raise ValueError("Response missing 'expanded_keywords' key")
    
    expanded_data = parsed_response['expanded_keywords']
    print(f"Found {len(expanded_data)} keyword groups")
    
except Exception as e:
    print(f"❌ JSON parsing failed: {e}")
    print("Raw response:")
    print(response)
    raise

In [ ]:
# Convert to DataFrame format
print("Converting to CSV format...")

rows = []
for item in expanded_data:
    original = item['original']
    category = item.get('category', 'variation')
    
    # Handle expanded keywords (could be string or list)
    expanded = item['expanded']
    if isinstance(expanded, str):
        expanded_list = [expanded]
    elif isinstance(expanded, list):
        expanded_list = expanded
    else:
        print(f"Warning: Unexpected expanded format for {original}: {expanded}")
        expanded_list = [str(expanded)]
    
    # Create row for each expanded keyword
    for exp_keyword in expanded_list:
        rows.append({
            'original': original,
            'expanded': exp_keyword.strip(),
            'category': category
        })

# Create DataFrame
df_keywords = pd.DataFrame(rows)

print(f"✅ Created DataFrame with {len(df_keywords)} rows")
print(f"Columns: {list(df_keywords.columns)}")
print(f"Unique categories: {df_keywords['category'].unique()}")

# Show sample
print("\n--- Sample of expanded keywords ---")
print(df_keywords.head(10))

In [ ]:
# Validate the output
print("Validating output...")

# Check for duplicates
duplicates = df_keywords.duplicated(subset=['original', 'expanded']).sum()
if duplicates > 0:
    print(f"⚠️  Found {duplicates} duplicate original-expanded pairs")
    df_keywords = df_keywords.drop_duplicates(subset=['original', 'expanded'])
    print(f"Removed duplicates, {len(df_keywords)} rows remaining")

# Check for empty values
empty_expanded = df_keywords['expanded'].isna().sum() + (df_keywords['expanded'] == '').sum()
if empty_expanded > 0:
    print(f"⚠️  Found {empty_expanded} empty expanded keywords")
    df_keywords = df_keywords[df_keywords['expanded'].notna() & (df_keywords['expanded'] != '')]
    print(f"Removed empty entries, {len(df_keywords)} rows remaining")

print(f"✅ Validation complete. Final dataset: {len(df_keywords)} keyword expansions")

In [ ]:
# Save the expanded keywords to CSV
save_csv_checkpoint(df_keywords, 'keywords_expanded.csv')
print("✅ Keywords expanded and saved to data/keywords_expanded.csv")

# Show final summary
print(f"\n📊 Summary:")
print(f"- Original keywords: {len(config['keywords'])}")
print(f"- Expanded keywords: {len(df_keywords)}")
print(f"- Categories: {', '.join(df_keywords['category'].unique())}")
print(f"- Avg expansions per original: {len(df_keywords) / len(config['keywords']):.1f}")

In [ ]:
# Optional: Show distribution by category
if len(df_keywords) > 0:
    print("\n📈 Distribution by category:")
    category_counts = df_keywords['category'].value_counts()
    for category, count in category_counts.items():
        print(f"- {category}: {count} keywords")